In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

## 데이터 로드
X = pd.read_csv('data/train.csv', index_col='Id')
X_test = pd.read_csv('data/test.csv', index_col='Id')

# 결측값 제거
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

# 결측 데이터가 존재하는 컬럼 삭제
missing_col = [col for col in X.columns if X[col].isnull().any()]

X.drop(missing_col, axis=1, inplace=True)
X_test.drop(missing_col, axis=1, inplace=True)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

def score_dataset(X_train, X_valid, y_train, y_valid) :
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid,preds)


### [Step 1] : 범주형 데이터 삭제하기

In [19]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop categorical variables):
17837.82570776256


In [20]:
print("Unique values in 'Condition2' column in training data:", X_train['Condition2'].unique())
print("\nUnique values in 'Condition2' column in validation data:", X_valid['Condition2'].unique())

Unique values in 'Condition2' column in training data: ['Norm' 'PosA' 'Feedr' 'PosN' 'Artery' 'RRAe']

Unique values in 'Condition2' column in validation data: ['Norm' 'RRAn' 'RRNn' 'Artery' 'Feedr' 'PosN']


### [Step 2] : 순서 인코딩

In [25]:
# 범주형 데이터 뽑아내기
object_cols = [ col for col in X_train.columns if X_train[col].dtype=='object']

print("object_cols" , object_cols)

# 순서 인코딩 할 수 있는 열 뽑기 -- 'RRAn' 'RRNn' 
good_label_cols = [col for col in object_cols if set(X_valid[col]).issubset(X_train[col])]
print("good_label_cols", good_label_cols)

bed_label_cols = list(set(object_cols)-set(good_label_cols))
print("bed_label_cols", bed_label_cols)

object_cols ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']
good_label_cols ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'PavedDrive', 'SaleType', 'SaleCondition']
bed_lable_cols ['Condition2', 'RoofMatl', 'Functional']


In [27]:
# X_train, X_valid 의 데이터를 순서 인코딩

from sklearn.preprocessing import OrdinalEncoder

# bed_label_cols remove
label_X_train = X_train.drop(bed_label_cols, axis=1)
label_X_valid = X_valid.drop(bed_label_cols, axis=1)

ord = OrdinalEncoder()
label_X_train[good_label_cols] = ord.fit_transform(X_train[good_label_cols])
label_X_valid[good_label_cols] = ord.transform(X_valid[good_label_cols])

print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE from Approach 2 (Ordinal Encoding):
17098.01649543379


In [28]:
# 범주형 데이터 열마다 고유한 값의 개수
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Print number of unique entries by column, in ascending order
sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('Utilities', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('LotShape', 4),
 ('LandContour', 4),
 ('ExterQual', 4),
 ('KitchenQual', 4),
 ('MSZoning', 5),
 ('LotConfig', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('HeatingQC', 5),
 ('Condition2', 6),
 ('RoofStyle', 6),
 ('Foundation', 6),
 ('Heating', 6),
 ('Functional', 6),
 ('SaleCondition', 6),
 ('RoofMatl', 7),
 ('HouseStyle', 8),
 ('Condition1', 9),
 ('SaleType', 9),
 ('Exterior1st', 15),
 ('Exterior2nd', 16),
 ('Neighborhood', 25)]

### [Step 3] : 범주형 변수의 카디널리티 조사하기

In [41]:
# 훈련 데이터에서 카디널리티가 10을 초과하는 범주형 변수는 몇 개인가요?
high_cardinality_numcols = 3

# 훈련 데이터에서 'Neighborhood' 변수를 원-핫 인코딩하기 위해 필요한 열의 수는 얼마인가요?
num_cols_neighborhood = 25


# 해당 열을 원-핫 인코딩으로 대체했을 때 데이터셋에 추가되는 항목(값)의 수는 얼마인가요?
# 10,000행 * 100개의 새 열 - 기존 열 1개 제거 → 1,000,000 - 10,000 = 990,000
OH_entries_added =10000 * 100 -10000 

OH_entries_added

# 해당 열을 순서 인코딩으로 대체했을 때 데이터셋에 추가되는 항목(값)의 수는 얼마인가요?
# 순서 인코딩은 기존 열을 정수 하나로 대체하므로, 추가되는 항목은 없음
abel_entries_added = 0


# Columns that will be one-hot encoded
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < 10]

# Columns that will be dropped from the dataset
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print('Categorical columns that will be one-hot encoded:', low_cardinality_cols)
print('\nCategorical columns that will be dropped from the dataset:', high_cardinality_cols)

Categorical columns that will be one-hot encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']

Categorical columns that will be dropped from the dataset: ['Neighborhood', 'Exterior1st', 'Exterior2nd']


### [Step 4] : 원-핫 인코딩

다음 코드 셀을 사용하여 X_train과 X_valid 데이터에 원-핫 인코딩(one-hot encoding) 을 적용하세요.
전처리된 결과는 각각 OH_X_train 과 OH_X_valid 라는 DataFrame에 저장해야 합니다.

데이터셋 내의 모든 범주형 열은 object_cols 라는 Python 리스트에 들어 있습니다.
하지만 원-핫 인코딩은 low_cardinality_cols 에 있는 열만 대상으로 수행하세요.

In [60]:
from sklearn.preprocessing import OneHotEncoder

print(object_cols)

print(low_cardinality_cols)

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols]))

OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# 범주형 열 제거
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# 원-핫 인코딩된 열 추가
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# 모든 컬럼이 문자열 유형인지 확인
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))




['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']
['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']
MAE from Approach 3 (One-Hot Encoding):
17525.345719178084
